# Daphnet Freezing of Gait (FoG) Detection — Improved Pipeline

## Overview
This notebook implements an improved pipeline for Freezing of Gait detection using the Daphnet dataset.

### Key Features:
1. 4th-order Butterworth bandpass filter (0.5-20 Hz)
2. Cubic spline interpolation for missing values
3. 4-second windows (256 samples @ 64 Hz) per literature
4. Majority voting (>50%) for window labelling
5. Feature selection via mutual information (k=60)
6. Threshold optimization via Youden's J
7. 3 Fusion techniques: Feature-level, Stacking (3 configurations), Weighted voting

### Stacking Configurations:
- **Stack 1**: LightGBM base learners + RandomForest meta-learner
- **Stack 2**: XGBoost base learners + LightGBM meta-learner
- **Stack 3**: LightGBM base learners + RandomForest meta-learner

In [ ]:
from __future__ import annotations

import sys, os, time, json, warnings, logging
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt
from scipy.optimize import minimize
from scipy.interpolate import CubicSpline
from joblib import Parallel, delayed
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import KNNImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif, VarianceThreshold
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve)
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

# Optional: XGBoost, LightGBM, SMOTE
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('Warning: XGBoost not available')

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print('Warning: LightGBM not available')

try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False
    print('Warning: SMOTE not available')

warnings.filterwarnings('ignore')

# Try to import project modules
try:
    PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
    sys.path.insert(0, str(PROJECT_ROOT))
    from loaders import DaphnetDatasetLoader
    from features.extractors import FeatureExtractor
    HAS_PROJECT_MODULES = True
except Exception as e:
    print(f'Warning: Project modules not available: {e}')
    HAS_PROJECT_MODULES = False

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('daphnet')

In [ ]:
# Dataset and preprocessing parameters
FS = 64
WINDOW_SEC = 4.0
WINDOW_SAMPLES = int(WINDOW_SEC * FS)  # 256
TRAIN_OVERLAP = 0.50
TEST_OVERLAP = 0.0
LABEL_THRESH = 0.50  # majority voting

# Filter parameters
BP_LOW, BP_HIGH, BP_ORDER = 0.5, 20.0, 4
NPERSEG = min(128, WINDOW_SAMPLES)

# Feature and model parameters
K_FEATURES = 60
SEED = 42
N_INNER_CV = 3
N_SEARCH_ITER = 20
ZERO_FOG_SUBJECTS = {4, 10}

# Sensor groups
SENSOR_COLS = {
    'ankle': ['acc_forward_ankle', 'acc_vertical_ankle', 'acc_lateral_ankle'],
    'thigh': ['acc_forward_thigh', 'acc_vertical_thigh', 'acc_lateral_thigh'],
    'trunk': ['acc_forward_trunk', 'acc_vertical_trunk', 'acc_lateral_trunk'],
}
ALL_ACC = [c for cols in SENSOR_COLS.values() for c in cols]

# Paths (adjust based on your environment)
PROJECT_ROOT = Path('/Users/davidperez/Documents/tesis/Seminario_II') if Path('/Users/davidperez/Documents/tesis/Seminario_II').exists() else Path.cwd()
DATASET_PATH = PROJECT_ROOT / 'Datasets' / 'Daphnet fog' / 'dataset'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'daphnet_improved_results'

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset path: {DATASET_PATH}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# Helper Functions

def interpolate_missing(data: np.ndarray) -> np.ndarray:
    """Fill NaN values per channel using cubic spline interpolation.
    Falls back to linear for short segments, ffill/bfill for edges."""
    data = data.copy()
    if data.ndim == 1:
        mask = np.isnan(data)
        if not mask.any():
            return data
        good = np.flatnonzero(~mask)
        bad = np.flatnonzero(mask)
        if len(good) == 0:
            data[:] = 0.0
            return data
        if len(good) >= 4:
            cs = CubicSpline(good, data[good], extrapolate=True)
            data[bad] = cs(bad)
        elif len(good) >= 2:
            data[bad] = np.interp(bad, good, data[good])
        else:
            data[mask] = data[good[0]]
        return data
    for col in range(data.shape[1]):
        data[:, col] = interpolate_missing(data[:, col])
    return data


def bandpass_filter(data: np.ndarray, fs: int = FS, low: float = BP_LOW,
                    high: float = BP_HIGH, order: int = BP_ORDER) -> np.ndarray:
    """Apply zero-phase Butterworth bandpass filter."""
    sos = butter(order, [low, high], btype='band', fs=fs, output='sos')
    if data.ndim == 1:
        return sosfiltfilt(sos, data).astype(np.float64)
    return np.column_stack([sosfiltfilt(sos, data[:, i]) for i in range(data.shape[1])])


def zscore_normalize(data: np.ndarray) -> np.ndarray:
    """Per-trial z-score normalization (robust: median / MAD)."""
    med = np.nanmedian(data, axis=0)
    mad = np.nanmedian(np.abs(data - med), axis=0) * 1.4826
    mad[mad < 1e-10] = 1.0
    return (data - med) / mad


def create_windows(signal: np.ndarray, labels: np.ndarray, win_samples: int,
                   overlap: float, label_threshold: float) -> Tuple[np.ndarray, np.ndarray]:
    """Sliding-window segmentation with majority-vote labelling."""
    step = max(1, int(win_samples * (1 - overlap)))
    n = len(signal)
    windows, win_labels = [], []
    for start in range(0, n - win_samples + 1, step):
        end = start + win_samples
        w = signal[start:end]
        lbl_slice = labels[start:end]
        if np.any(np.isnan(w)):
            continue
        fog_ratio = np.mean(lbl_slice)
        win_labels.append(1 if fog_ratio > label_threshold else 0)
        windows.append(w)
    if not windows:
        return np.empty((0, win_samples, signal.shape[1])), np.empty(0)
    return np.array(windows), np.array(win_labels)


def youden_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    """Find optimal threshold via Youden's J statistic."""
    if len(np.unique(y_true)) < 2:
        return 0.5
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    j = tpr - fpr
    return float(thresholds[np.argmax(j)])


def compute_metrics(y_true, y_pred, y_prob=None) -> Dict[str, float]:
    """Compute classification metrics."""
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_acc': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn),
    }
    if y_prob is not None and len(np.unique(y_true)) == 2:
        metrics['auc'] = roc_auc_score(y_true, y_prob)
    return metrics

In [ ]:
# Data Loading and Preprocessing

def load_and_preprocess() -> Dict[int, Dict]:
    """Load Daphnet data, preprocess per subject/trial. Returns dict[subject_id] -> {windows, labels}."""
    if not HAS_PROJECT_MODULES:
        log.error('Project modules required for data loading')
        return {}
    
    log.info('Loading Daphnet dataset from %s', DATASET_PATH)
    loader = DaphnetDatasetLoader(str(DATASET_PATH))
    df = loader.load_all_data(verbose=False)

    # Filter out annotation=0 (non-experiment)
    n_before = len(df)
    df = df[df['annotation'] != 0].reset_index(drop=True)
    log.info('Filtered annotation=0: %d -> %d samples', n_before, len(df))

    # Binary labels: annotation 2 -> FoG (1), annotation 1 -> No FoG (0)
    df['fog_label'] = (df['annotation'] == 2).astype(int)

    subjects_data = {}
    for sid in sorted(df['subject_id'].unique()):
        sub_df = df[df['subject_id'] == sid]
        all_windows, all_labels = [], []

        for rid in sorted(sub_df['run_id'].unique()):
            trial = sub_df[sub_df['run_id'] == rid]
            signal = trial[ALL_ACC].values.astype(np.float64)
            labels_raw = trial['fog_label'].values.astype(np.float64)

            # Interpolate missing values before filtering
            signal = interpolate_missing(signal)
            labels_raw = np.nan_to_num(labels_raw, nan=0.0)

            # Bandpass filter
            signal = bandpass_filter(signal, FS)

            # Add magnitude per sensor
            mags = []
            for sensor_name, cols in SENSOR_COLS.items():
                idxs = [ALL_ACC.index(c) for c in cols]
                mag = np.sqrt(np.sum(signal[:, idxs] ** 2, axis=1, keepdims=True))
                mags.append(mag)
            signal = np.hstack([signal, np.hstack(mags)])  # 9 + 3 = 12 channels

            # Z-score normalize per trial
            signal = zscore_normalize(signal)

            # Create windows
            w, l = create_windows(signal, labels_raw, WINDOW_SAMPLES, TRAIN_OVERLAP, LABEL_THRESH)
            if len(w) > 0:
                all_windows.append(w)
                all_labels.append(l)

        if all_windows:
            subjects_data[sid] = {
                'windows': np.concatenate(all_windows, axis=0),
                'labels': np.concatenate(all_labels, axis=0),
            }
            n_fog = int(np.sum(subjects_data[sid]['labels']))
            log.info('  Subject S%02d: %d windows (%d FoG, %d NoFoG)',
                     sid, len(subjects_data[sid]['labels']), n_fog,
                     len(subjects_data[sid]['labels']) - n_fog)

    return subjects_data

In [ ]:
# Feature Extraction

def extract_features_for_subject(windows: np.ndarray, fs: int = FS) -> pd.DataFrame:
    """Extract features from all windows of a subject."""
    if not HAS_PROJECT_MODULES:
        log.error('Project modules required for feature extraction')
        return pd.DataFrame()
    
    extractor = FeatureExtractor(
        sampling_rate=fs,
        extract_time=True,
        extract_frequency=True,
        extract_wavelet=True,
        extract_nonlinear=False,  # too slow for pipeline
    )
    rows = []
    for i in range(len(windows)):
        feats = extractor.extract_from_window(windows[i], include_magnitude=False,
                                               channel_groups=None)
        rows.append(feats)
    return pd.DataFrame(rows)


def extract_all_features(subjects_data: Dict) -> Dict[int, Dict]:
    """Extract features for all subjects in parallel."""
    log.info('Extracting features for %d subjects (parallel)...', len(subjects_data))

    def _extract(sid):
        feat_df = extract_features_for_subject(subjects_data[sid]['windows'])
        return sid, feat_df

    results = Parallel(n_jobs=-1, verbose=0)(
        delayed(_extract)(sid) for sid in tqdm(subjects_data.keys(), desc='Feature extraction')
    )

    features = {}
    for sid, feat_df in results:
        features[sid] = {
            'X': feat_df,
            'y': subjects_data[sid]['labels'],
        }
        log.info('  S%02d: %d windows, %d features', sid, len(feat_df), feat_df.shape[1])
    return features

In [ ]:
# Preprocessing and Feature Selection

def get_sensor_features(X: pd.DataFrame, sensor_name: str) -> pd.DataFrame:
    """Extract features for a specific sensor."""
    sensor_prefixes = {
        'ankle': 'ankle',
        'thigh': 'thigh',
        'trunk': 'trunk',
    }
    prefix = sensor_prefixes.get(sensor_name, sensor_name)
    matching_cols = [col for col in X.columns if prefix in col.lower()]
    return X[matching_cols] if matching_cols else pd.DataFrame()


def preprocess_features(X_train: pd.DataFrame, X_test: pd.DataFrame,
                        y_train: np.ndarray, k: int = K_FEATURES):
    """Clean, scale, select features. Returns processed arrays + selector."""
    # Replace inf
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_test = X_test.replace([np.inf, -np.inf], np.nan)

    # Variance threshold (remove constants)
    vt = VarianceThreshold(threshold=0.0)
    X_train_vt = pd.DataFrame(vt.fit_transform(X_train), columns=X_train.columns[vt.get_support()])
    X_test_vt = pd.DataFrame(vt.transform(X_test), columns=X_train_vt.columns)

    # Impute
    imputer = KNNImputer(n_neighbors=5)
    X_train_imp = pd.DataFrame(imputer.fit_transform(X_train_vt), columns=X_train_vt.columns)
    X_test_imp = pd.DataFrame(imputer.transform(X_test_vt), columns=X_train_vt.columns)

    # Scale
    scaler = RobustScaler()
    X_train_sc = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X_train_imp.columns)
    X_test_sc = pd.DataFrame(scaler.transform(X_test_imp), columns=X_train_imp.columns)

    # Feature selection
    k_actual = min(k, X_train_sc.shape[1])
    selector = SelectKBest(mutual_info_classif, k=k_actual)
    X_train_sel = selector.fit_transform(X_train_sc, y_train)
    X_test_sel = selector.transform(X_test_sc)

    selected_cols = X_train_sc.columns[selector.get_support()].tolist()

    return X_train_sel, X_test_sel, selected_cols, {'vt': vt, 'imputer': imputer, 'scaler': scaler, 'selector': selector}


def prepare_fold(features: Dict, test_sid: int):
    """Prepare train/test split for one LOSO fold."""
    X_trains, y_trains = [], []
    for sid, data in features.items():
        if sid == test_sid:
            continue
        X_trains.append(data['X'])
        y_trains.append(data['y'])

    X_train = pd.concat(X_trains, ignore_index=True)
    y_train = np.concatenate(y_trains)
    X_test = features[test_sid]['X'].copy()
    y_test = features[test_sid]['y'].copy()
    return X_train, y_train, X_test, y_test

In [ ]:
# Stacking Model Functions (3 Configurations)

def _build_sensor_model(X_tr, y_tr, X_te, stacking_config=1):
    """Train a per-sensor model with configurable stacking.
    
    stacking_config:
        1: LightGBM base learner + RandomForest meta-learner
        2: XGBoost base learner + LightGBM meta-learner
        3: LightGBM base learner + RandomForest meta-learner (same as 1)
    
    Returns (test_probs, train_oof_probs).
    """
    from sklearn.model_selection import StratifiedKFold as SKF

    X_tr = np.asarray(X_tr, dtype=np.float64)
    X_te = np.asarray(X_te, dtype=np.float64)

    # Generate OOF predictions for stacking (no leakage)
    oof_probs = np.zeros(len(y_tr))
    inner_cv = SKF(n_splits=3, shuffle=True, random_state=SEED)

    # Select base learner based on config
    if stacking_config == 1:
        # Stack 1: LightGBM base
        if not HAS_LGBM:
            log.warning('LightGBM not available, falling back to RandomForest')
            base_clf = RandomForestClassifier(n_estimators=200, max_depth=20,
                                              class_weight='balanced', random_state=SEED, n_jobs=-1)
        else:
            base_clf = LGBMClassifier(n_estimators=200, max_depth=20, num_leaves=31,
                                     learning_rate=0.1, class_weight='balanced',
                                     random_state=SEED, verbose=-1, n_jobs=-1)
    elif stacking_config == 2:
        # Stack 2: XGBoost base
        if not HAS_XGB:
            log.warning('XGBoost not available, falling back to RandomForest')
            base_clf = RandomForestClassifier(n_estimators=200, max_depth=20,
                                              class_weight='balanced', random_state=SEED, n_jobs=-1)
        else:
            base_clf = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                    subsample=0.8, colsample_bytree=0.8,
                                    eval_metric='logloss', random_state=SEED,
                                    n_jobs=-1, verbosity=0)
    else:  # stacking_config == 3
        # Stack 3: LightGBM base (same as 1)
        if not HAS_LGBM:
            log.warning('LightGBM not available, falling back to RandomForest')
            base_clf = RandomForestClassifier(n_estimators=200, max_depth=20,
                                              class_weight='balanced', random_state=SEED, n_jobs=-1)
        else:
            base_clf = LGBMClassifier(n_estimators=200, max_depth=20, num_leaves=31,
                                     learning_rate=0.1, class_weight='balanced',
                                     random_state=SEED, verbose=-1, n_jobs=-1)

    for tr_idx, val_idx in inner_cv.split(X_tr, y_tr):
        X_f, y_f = X_tr[tr_idx], y_tr[tr_idx]
        X_v = X_tr[val_idx]

        # SMOTE on inner train
        if HAS_SMOTE and np.sum(y_f == 1) >= 6:
            try:
                k_n = min(5, int(np.sum(y_f == 1)) - 1)
                sm = SMOTE(random_state=SEED, k_neighbors=k_n)
                X_f, y_f = sm.fit_resample(X_f, y_f)
            except Exception:
                pass

        clf_inner = base_clf.__class__(**base_clf.get_params())
        clf_inner.fit(X_f, y_f)
        oof_probs[val_idx] = clf_inner.predict_proba(X_v)[:, 1]

    # Train final model on full training set for test predictions
    X_tr_sm, y_tr_sm = X_tr, y_tr
    if HAS_SMOTE and np.sum(y_tr == 1) >= 6:
        try:
            k_n = min(5, int(np.sum(y_tr == 1)) - 1)
            sm = SMOTE(random_state=SEED, k_neighbors=k_n)
            X_tr_sm, y_tr_sm = sm.fit_resample(X_tr, y_tr)
        except Exception:
            pass

    clf_full = base_clf.__class__(**base_clf.get_params())
    clf_full.fit(X_tr_sm, y_tr_sm)
    test_probs = clf_full.predict_proba(X_te)[:, 1]

    return test_probs, oof_probs

In [ ]:
# Fusion Evaluation (Feature-level, 3x Stacking, Weighted voting)

def run_fusion_evaluation(features: Dict, stacking_config=1):
    """Run 3 fusion techniques evaluation including stacking config. No test-set leakage.
    
    stacking_config: 1, 2, or 3 (see _build_sensor_model)
    """
    subjects = sorted(features.keys())
    fusion_results = {'feature_level': [], 'stacking': [], 'weighted_voting': []}

    for test_sid in tqdm(subjects, desc=f'Fusion LOSO (Stack {stacking_config})'):
        X_train, y_train, X_test, y_test = prepare_fold(features, test_sid)
        has_both = len(np.unique(y_test)) == 2

        # ── Fusion 1: Feature-Level ──
        X_tr_p, X_te_p, _, _ = preprocess_features(X_train, X_test, y_train, k=K_FEATURES)

        X_tr_sm, y_tr_sm = X_tr_p, y_train
        if HAS_SMOTE and np.sum(y_train == 1) >= 6:
            try:
                k_n = min(5, np.sum(y_train == 1) - 1)
                sm = SMOTE(random_state=SEED, k_neighbors=k_n)
                X_tr_sm, y_tr_sm = sm.fit_resample(X_tr_p, y_train)
            except Exception:
                pass

        clf_fl = RandomForestClassifier(n_estimators=300, max_depth=20, class_weight='balanced',
                                         random_state=SEED, n_jobs=-1)
        clf_fl.fit(X_tr_sm, y_tr_sm)
        y_prob_fl = clf_fl.predict_proba(X_te_p)[:, 1]

        # Threshold on validation split (not test)
        if len(y_train) > 20 and len(np.unique(y_train)) > 1:
            _, X_val_fl, _, y_val_fl = train_test_split(X_tr_p, y_train, test_size=0.2,
                                                         stratify=y_train, random_state=SEED)
            y_val_prob_fl = clf_fl.predict_proba(X_val_fl)[:, 1]
            thr_fl = youden_threshold(y_val_fl, y_val_prob_fl)
        else:
            thr_fl = 0.5

        y_pred_fl = (y_prob_fl >= thr_fl).astype(int)
        m_fl = compute_metrics(y_test, y_pred_fl, y_prob_fl)
        m_fl['subject'] = test_sid
        m_fl['has_both_classes'] = has_both
        fusion_results['feature_level'].append(m_fl)

        # ── Fusion 2 & 3: Per-sensor models with OOF predictions ──
        sensor_oof = {}     # OOF predictions for stacking (no leakage)
        sensor_test = {}    # Test predictions

        for sensor_name in ['ankle', 'thigh', 'trunk']:
            X_tr_s = get_sensor_features(X_train, sensor_name)
            X_te_s = get_sensor_features(X_test, sensor_name)

            if X_tr_s.shape[1] == 0:
                continue

            X_tr_s = X_tr_s.replace([np.inf, -np.inf], np.nan)
            X_te_s = X_te_s.replace([np.inf, -np.inf], np.nan)

            vt = VarianceThreshold(threshold=0.0)
            try:
                X_tr_s_vt = pd.DataFrame(vt.fit_transform(X_tr_s))
                X_te_s_vt = pd.DataFrame(vt.transform(X_te_s))
            except Exception:
                continue
            if X_tr_s_vt.shape[1] == 0:
                continue

            imp = KNNImputer(n_neighbors=5)
            X_tr_s_i = imp.fit_transform(X_tr_s_vt)
            X_te_s_i = imp.transform(X_te_s_vt)

            sc = RobustScaler()
            X_tr_s_sc = sc.fit_transform(X_tr_s_i)
            X_te_s_sc = sc.transform(X_te_s_i)

            k_s = min(20, X_tr_s_sc.shape[1])
            sel = SelectKBest(mutual_info_classif, k=k_s)
            X_tr_s_sel = sel.fit_transform(X_tr_s_sc, y_train)
            X_te_s_sel = sel.transform(X_te_s_sc)

            test_probs, oof_probs = _build_sensor_model(X_tr_s_sel, y_train, X_te_s_sel, stacking_config)
            sensor_test[sensor_name] = test_probs
            sensor_oof[sensor_name] = oof_probs

        if len(sensor_test) < 2:
            continue

        sensor_names = list(sensor_test.keys())
        P_test = np.column_stack([sensor_test[s] for s in sensor_names])
        P_oof = np.column_stack([sensor_oof[s] for s in sensor_names])

        # ── Fusion 2: Stacking with OOF predictions (no leakage) ──
        # Select meta-learner based on config
        if stacking_config == 1:
            # Stack 1: RandomForest meta-learner
            meta_clf = RandomForestClassifier(n_estimators=100, max_depth=10,
                                             class_weight='balanced', random_state=SEED, n_jobs=-1)
        elif stacking_config == 2:
            # Stack 2: LightGBM meta-learner
            if not HAS_LGBM:
                log.warning('LightGBM not available for meta-learner, using LogisticRegression')
                meta_clf = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
            else:
                meta_clf = LGBMClassifier(n_estimators=100, max_depth=5, num_leaves=31,
                                         learning_rate=0.1, class_weight='balanced',
                                         random_state=SEED, verbose=-1)
        else:  # stacking_config == 3
            # Stack 3: RandomForest meta-learner (same as 1)
            meta_clf = RandomForestClassifier(n_estimators=100, max_depth=10,
                                             class_weight='balanced', random_state=SEED, n_jobs=-1)

        meta_clf.fit(P_oof, y_train)  # trained on OOF, not in-sample
        y_prob_stack = meta_clf.predict_proba(P_test)[:, 1]

        # Threshold on OOF validation
        y_oof_stack = meta_clf.predict_proba(P_oof)[:, 1]
        thr_st = youden_threshold(y_train, y_oof_stack)
        y_pred_stack = (y_prob_stack >= thr_st).astype(int)
        m_st = compute_metrics(y_test, y_pred_stack, y_prob_stack)
        m_st['subject'] = test_sid
        m_st['has_both_classes'] = has_both
        m_st['stacking_config'] = stacking_config
        fusion_results['stacking'].append(m_st)

        # ── Fusion 3: Weighted Voting — weights optimized on OOF (not test) ──
        def neg_f1_weighted(w):
            w = np.abs(w)
            w = w / (w.sum() + 1e-12)
            fused = P_oof @ w
            preds = (fused >= 0.5).astype(int)
            return -f1_score(y_train, preds, zero_division=0)

        w0 = np.ones(len(sensor_names)) / len(sensor_names)
        try:
            res = minimize(neg_f1_weighted, w0, method='Nelder-Mead',
                          options={'maxiter': 200})
            w_opt = np.abs(res.x)
            w_opt = w_opt / (w_opt.sum() + 1e-12)
        except Exception:
            w_opt = w0

        y_prob_wv = P_test @ w_opt
        # Threshold on OOF validation
        y_oof_wv = P_oof @ w_opt
        thr_wv = youden_threshold(y_train, y_oof_wv)
        y_pred_wv = (y_prob_wv >= thr_wv).astype(int)
        m_wv = compute_metrics(y_test, y_pred_wv, y_prob_wv)
        m_wv['subject'] = test_sid
        m_wv['has_both_classes'] = has_both
        m_wv['weights'] = {s: round(w, 3) for s, w in zip(sensor_names, w_opt)}
        fusion_results['weighted_voting'].append(m_wv)

    return fusion_results

In [ ]:
# Results Reporting

def aggregate_results(results_list: List[Dict], name: str = '') -> Dict:
    """Aggregate LOSO fold results, excluding monoclase folds for F1/recall."""
    evaluable = [r for r in results_list if r.get('has_both_classes', True)]
    all_folds = results_list

    # Aggregate confusion matrix across ALL folds
    cm_total = np.zeros((2, 2), dtype=int)
    for r in all_folds:
        cm_total += np.array([[r['tn'], r['fp']], [r['fn'], r['tp']]])

    tn, fp, fn, tp = cm_total.ravel()
    total = tn + fp + fn + tp

    agg = {
        'name': name,
        'n_folds_total': len(all_folds),
        'n_folds_evaluable': len(evaluable),
        'accuracy_agg': (tp + tn) / total if total > 0 else 0,
        'precision_agg': tp / (tp + fp) if (tp + fp) > 0 else 0,
        'recall_agg': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity_agg': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'f1_agg': 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0,
        'cm_total': cm_total.tolist(),
    }

    # Mean of evaluable folds
    if evaluable:
        for metric in ['f1', 'recall', 'precision', 'specificity', 'balanced_acc']:
            vals = [r[metric] for r in evaluable if metric in r]
            agg[f'{metric}_mean'] = np.mean(vals) if vals else 0
        auc_vals = [r['auc'] for r in evaluable if 'auc' in r]
        agg['auc_mean'] = np.mean(auc_vals) if auc_vals else 0

    return agg


def print_fusion_summary(all_fusion_results: Dict):
    """Print summary of all fusion techniques."""
    print(f"\n{'='*80}")
    print(f"  FUSION TECHNIQUES COMPARISON (Daphnet)")
    print(f"{'='*80}")
    
    for fusion_name, results in all_fusion_results.items():
        if not results:
            continue
        agg = aggregate_results(results, fusion_name)
        print(f"\n{fusion_name.upper()}:")
        print(f"  F1 (agg): {agg['f1_agg']:.4f} | F1 (mean): {agg.get('f1_mean', 0):.4f}")
        print(f"  Recall: {agg.get('recall_mean', 0):.4f} | Precision: {agg.get('precision_mean', 0):.4f}")
        print(f"  Specificity: {agg.get('specificity_mean', 0):.4f} | BalAcc: {agg.get('balanced_acc_mean', 0):.4f}")
        print(f"  Folds: {agg['n_folds_evaluable']}/{agg['n_folds_total']}")

In [ ]:
# Main Execution

# NOTE: Run this cell to execute the full pipeline
# WARNING: Full execution may take 30+ minutes depending on your machine

if __name__ == '__main__' or True:  # Allow notebook execution
    t0 = time.time()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print('=' * 90)
    print('  DAPHNET FoG DETECTION — IMPROVED PIPELINE')
    print('=' * 90)
    print(f'  Window: {WINDOW_SEC}s ({WINDOW_SAMPLES} samples)')
    print(f'  Bandpass: {BP_LOW}-{BP_HIGH} Hz')
    print(f'  Label threshold: {LABEL_THRESH} (majority voting)')
    print(f'  Feature selection: top {K_FEATURES} (mutual information)')
    print(f'  Threshold optimisation: Youden\'s J')
    print()

    # Step 1: Load & preprocess
    if HAS_PROJECT_MODULES:
        subjects_data = load_and_preprocess()

        # Step 2: Extract features
        features = extract_all_features(subjects_data)

        # Step 3: Run fusion evaluation for each stacking configuration
        all_fusion_results = {}
        for config in [1, 2, 3]:
            print(f'\nRunning Stacking Configuration {config}...')
            fusion_results = run_fusion_evaluation(features, stacking_config=config)
            all_fusion_results[f'stacking_config_{config}'] = fusion_results

        # Step 4: Print results
        print_fusion_summary(all_fusion_results['stacking_config_1'])
        
        elapsed = time.time() - t0
        print(f'\nTotal execution time: {elapsed/60:.1f} minutes')
    else:
        print('ERROR: Project modules required. Please ensure you have access to:')
        print('  - loaders.DaphnetDatasetLoader')
        print('  - features.extractors.FeatureExtractor')